# Build Streamflow Training Parquet

This notebook reads the PostgreSQL `variable` table, discovers every available sensor variable, converts long-format sensor readings into hourly model features, builds 7-day lookback samples, and saves a Parquet file for streamflow prediction.

The target is next-hour `streamflow`/`discharge`. The input window uses the previous hourly rows for every variable listed in `public.variable`, plus time features.


## 1. Configure Connection and Training Settings

In [18]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sqlalchemy import create_engine, text
import pyarrow as pa
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

# Database settings from README.md / docker-compose.yml.
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5433")
DB_NAME = os.getenv("DB_NAME", "database")
DB_USER = os.getenv("DB_USER", "admin")
DB_PASSWORD = os.getenv("DB_PASSWORD", "password")

DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)

# Streamflow/discharge exists at site_id 3 and 4 in this database.
TARGET_SITE_ID = 3

# Optional source-site overrides. Any variable not listed here is loaded from the target
# site when available, otherwise from the site with the most rows for that variable.
VARIABLE_SOURCE_SITE_IDS = {
    "streamflow": TARGET_SITE_ID,
    "water_temp": TARGET_SITE_ID,
    "oxygen_dissolved": TARGET_SITE_ID,
    "air_temp": 2,
    "precipitation": 2,
}

# Terms used to identify the target variable in the variable table.
TARGET_VARIABLE_MATCH_TERMS = ["streamflow", "stream flow", "discharge"]

# Seven days of hourly history = 168 rows.
LOOKBACK_HOURS = 168

# Set to an integer to build a small inspection sample, or None for the full dataset.
SAMPLE_TRAINING_ROWS = None

# Set to an integer to drop rows before imputation when too many sensor features
# are missing. None keeps rows with observed streamflow and imputes feature gaps.
MAX_MISSING_MAIN_PREDICTORS = None

# Use hour-ending bins so 00:15, 00:30, 00:45, and 01:00 all map to 01:00.
# If you prefer hour-start bins, change to "hour_start".
HOURLY_BINNING = "hour_end"

OUTPUT_PARQUET = Path(f"streamflow_training_site_{TARGET_SITE_ID}_full.parquet")

# Populated inside build_training_parquet_for_site after reading public.variable.
MAIN_PREDICTOR_COLUMNS = []
LOOKBACK_FEATURE_COLUMNS = []


## 2. Check Available Variables

In [19]:
def list_database_variables(engine):
    """Return variable IDs, names, codes, and units from the database."""
    query = text("""
        SELECT
            v.variable_id,
            v.variable_code,
            v.name AS variable_name,
            v.variable_type,
            u.name AS unit_name,
            u.symbol AS unit_symbol
        FROM public.variable v
        LEFT JOIN public.unit u ON v.unit_id = u.unit_id
        ORDER BY v.variable_id
    """)
    return pd.read_sql_query(query, engine)


variables_df = list_database_variables(engine)
display(variables_df)

,variable_id,variable_code,variable_name,variable_type,unit_name,unit_symbol
0,1,SnowDepth,Snow depth,Hydrology,centimeter,cm
1,2,WaterTemp,Water Temperature,Water Quality,degree Celsius,C
2,3,ODO,"Oxygen, dissolved",Water Quality,milligrams per liter,mg/L
3,4,Discharge,Discharge,Hydrology,cubic feet per second,cfs
4,5,AirTemp,Air Temperature,Climate,degree Celsius,C
5,6,Precip,Precipitation,Hydrology,centimeter,cm


## 3. Helper Functions

In [20]:
def month_to_season(month):
    """Encode season numerically: winter=0, spring=1, summer=2, fall=3."""
    if month in (12, 1, 2):
        return 0
    if month in (3, 4, 5):
        return 1
    if month in (6, 7, 8):
        return 2
    return 3


def check_parquet_support():
    """Fail early with a clear message if PyArrow is not available."""
    try:
        import pyarrow  # noqa: F401
        import pyarrow.parquet  # noqa: F401
    except ImportError as exc:
        raise ImportError(
            "This notebook needs PyArrow before it can save the final file. "
            "Install it with: python3 -m pip install pyarrow"
        ) from exc


def write_parquet_with_pyarrow(df, output_path):
    """Write Parquet through PyArrow directly, bypassing pandas' parquet wrapper."""
    table = pa.Table.from_pandas(df, preserve_index=False)
    pq.write_table(table, output_path)


def read_parquet_with_pyarrow(input_path):
    """Read Parquet through PyArrow directly, bypassing pandas' parquet wrapper."""
    return pq.read_table(input_path).to_pandas()


def to_snake_case(value):
    """Convert database names/codes to stable feature column names."""
    value = "" if pd.isna(value) else str(value)
    value = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", value)
    value = re.sub(r"[^0-9A-Za-z]+", "_", value)
    return value.strip("_").lower()


def feature_name_for_variable(row, target_match_terms):
    searchable = f"{row.get('variable_name', '')} {row.get('variable_code', '')}".lower()
    if any(term.lower() in searchable for term in target_match_terms):
        return "streamflow"

    raw_name = row.get("variable_code") or row.get("variable_name")
    feature_name = to_snake_case(raw_name)
    aliases = {
        "air_temp": "air_temp",
        "air_temperature": "air_temp",
        "water_temp": "water_temp",
        "water_temperature": "water_temp",
        "precip": "precipitation",
        "odo": "oxygen_dissolved",
        "oxygen_dissolved": "oxygen_dissolved",
        "snow_depth": "snow_depth",
    }
    return aliases.get(feature_name, feature_name)


def resolve_variable_mapping(variables, target_match_terms):
    """Map every raw database variable_id to a feature name from public.variable."""
    variables = variables.copy()
    variables["feature_name"] = variables.apply(
        lambda row: feature_name_for_variable(row, target_match_terms), axis=1
    )

    if "streamflow" not in set(variables["feature_name"]):
        raise ValueError(
            "Could not identify the streamflow target in public.variable. "
            "Update TARGET_VARIABLE_MATCH_TERMS."
        )

    duplicate_features = variables[variables.duplicated("feature_name", keep=False)]
    if not duplicate_features.empty:
        variables.loc[duplicate_features.index, "feature_name"] = duplicate_features.apply(
            lambda row: f"{row['feature_name']}_{int(row['variable_id'])}", axis=1
        )

    variables = variables.sort_values("variable_id").reset_index(drop=True)
    variable_id_to_name = dict(zip(variables["variable_id"].astype(int), variables["feature_name"]))
    feature_to_variable_ids = (
        variables.groupby("feature_name")["variable_id"]
        .apply(lambda ids: [int(variable_id) for variable_id in ids])
        .to_dict()
    )
    feature_columns = variables["feature_name"].tolist()

    print("Variables available in public.variable and included as features:")
    display(variables[["variable_id", "variable_code", "variable_name", "feature_name", "unit_name", "unit_symbol"]])
    print(f"Feature columns: {feature_columns}")

    return variable_id_to_name, feature_to_variable_ids, feature_columns


def list_variable_site_availability(engine):
    """Return row counts for every site/variable combination in datastream."""
    query = text("""
        SELECT
            d.site_id,
            s.site_code,
            d.variable_id,
            v.variable_code,
            v.name AS variable_name,
            COUNT(*) AS row_count,
            MIN(d.datetime_utc) AS min_datetime,
            MAX(d.datetime_utc) AS max_datetime
        FROM public.datastream d
        JOIN public.variable v ON d.variable_id = v.variable_id
        JOIN public.site s ON d.site_id = s.site_id
        GROUP BY d.site_id, s.site_code, d.variable_id, v.variable_code, v.name
        ORDER BY d.variable_id, d.site_id
    """)
    return pd.read_sql_query(query, engine)


def choose_source_site(feature_name, variable_ids, target_site_id, variable_source_site_ids, availability):
    """Choose the source site for one feature, honoring overrides when present."""
    if feature_name in variable_source_site_ids:
        return int(variable_source_site_ids[feature_name])

    candidates = availability[availability["variable_id"].isin(variable_ids)].copy()
    if candidates.empty:
        raise ValueError(f"No datastream rows found for feature {feature_name}.")

    target_candidates = candidates[candidates["site_id"] == target_site_id]
    if not target_candidates.empty:
        return int(target_site_id)

    best = candidates.sort_values(["row_count", "site_id"], ascending=[False, True]).iloc[0]
    return int(best["site_id"])


def load_datastream_for_site(
    engine,
    target_site_id,
    variable_id_to_name,
    feature_to_variable_ids,
    feature_columns,
    variable_source_site_ids,
    availability,
):
    """Load long-format rows for all discovered features."""
    frames = []

    for feature_name in feature_columns:
        variable_ids = feature_to_variable_ids[feature_name]
        source_site_id = choose_source_site(
            feature_name, variable_ids, target_site_id, variable_source_site_ids, availability
        )
        query = text("""
            SELECT
                :target_site_id AS site_id,
                d.site_id AS source_site_id,
                d.variable_id,
                d.datetime_utc AS datetime,
                d.value
            FROM public.datastream d
            WHERE d.site_id = :source_site_id
              AND d.variable_id = ANY(:variable_ids)
            ORDER BY d.datetime_utc, d.variable_id
        """)
        df = pd.read_sql_query(
            query,
            engine,
            params={
                "target_site_id": target_site_id,
                "source_site_id": source_site_id,
                "variable_ids": variable_ids,
            },
        )
        if df.empty:
            raise ValueError(
                f"No rows found for {feature_name}: source_site_id={source_site_id}, "
                f"variable_ids={variable_ids}. Check VARIABLE_SOURCE_SITE_IDS."
            )

        negative_value_count = int((df["value"] < 0).sum())
        if negative_value_count:
            df.loc[df["value"] < 0, "value"] = np.nan
            print(f"Converted {negative_value_count:,} negative {feature_name} readings to missing values")

        df["datetime"] = pd.to_datetime(df["datetime"])
        df["variable_name"] = df["variable_id"].map(variable_id_to_name)
        frames.append(df)
        print(f"Loaded {len(df):,} rows for {feature_name} from source_site_id={source_site_id}")

    return pd.concat(frames, ignore_index=True)


def assign_hourly_bucket(datetime_series, binning="hour_end"):
    """Assign each timestamp to a consistent hourly bucket."""
    if binning == "hour_end":
        return datetime_series.dt.ceil("h")
    if binning == "hour_start":
        return datetime_series.dt.floor("h")
    raise ValueError("HOURLY_BINNING must be 'hour_end' or 'hour_start'.")


def long_to_hourly_wide(df, feature_columns, binning="hour_end"):
    """Convert long 15-minute readings to hourly wide rows using last available reading per hour."""
    working = df.copy()
    working["hourly_datetime"] = assign_hourly_bucket(working["datetime"], binning=binning)

    # Keep the last non-missing reading inside each variable/hour bucket.
    working = working.dropna(subset=["value"]).sort_values(["variable_name", "hourly_datetime", "datetime"])
    hourly_long = working.groupby(["hourly_datetime", "variable_name"], as_index=False).tail(1)

    wide = hourly_long.pivot_table(
        index="hourly_datetime",
        columns="variable_name",
        values="value",
        aggfunc="last",
    ).reset_index()
    wide.columns.name = None
    target_site_id = int(working["site_id"].iloc[0])
    wide = wide.rename(columns={"hourly_datetime": "datetime"}).sort_values("datetime")

    # Reindex to a complete hourly timeline for the site so gaps are explicit.
    full_hours = pd.date_range(wide["datetime"].min(), wide["datetime"].max(), freq="h")
    wide = wide.set_index("datetime").reindex(full_hours).rename_axis("datetime").reset_index()
    wide["site_id"] = target_site_id

    for col in feature_columns:
        if col not in wide.columns:
            wide[col] = np.nan

    return wide[["site_id", "datetime"] + feature_columns]


def add_time_features(df):
    """Add year, month, day, hour, and encoded season from datetime."""
    out = df.copy()
    out["year"] = out["datetime"].dt.year
    out["month"] = out["datetime"].dt.month
    out["day"] = out["datetime"].dt.day
    out["hour"] = out["datetime"].dt.hour
    out["season"] = out["month"].map(month_to_season)
    return out


def drop_sparse_rows_and_impute(df, feature_columns, max_missing_main_predictors):
    """Drop very sparse rows, then impute remaining predictor gaps within the site."""
    out = df.copy().sort_values(["site_id", "datetime"])

    # Preserve observed streamflow separately; targets should be real observations, not imputed labels.
    out["streamflow_observed"] = out["streamflow"]

    if max_missing_main_predictors is not None:
        missing_count = out[feature_columns].isna().sum(axis=1)
        out = out.loc[missing_count <= max_missing_main_predictors].copy()

    # Impute remaining feature gaps using time interpolation, then edge fill.
    out = out.set_index("datetime")
    out[feature_columns] = (
        out.groupby("site_id", group_keys=False)[feature_columns]
        .apply(lambda group: group.interpolate(method="time").ffill().bfill())
    )
    out = out.reset_index().sort_values(["site_id", "datetime"])

    return out


def build_supervised_samples(df, lookback_hours, feature_columns, max_samples=None):
    """Flatten lookback windows into rows with next-hour streamflow targets."""
    rows = []
    expected_step = pd.Timedelta(hours=1)

    for site_id, site_df in df.groupby("site_id", sort=True):
        site_df = site_df.sort_values("datetime").reset_index(drop=True)
        times = site_df["datetime"]

        for target_idx in range(lookback_hours, len(site_df)):
            target_streamflow = site_df.loc[target_idx, "streamflow_observed"]
            if pd.isna(target_streamflow):
                continue

            window = site_df.iloc[target_idx - lookback_hours:target_idx]
            window_times = times.iloc[target_idx - lookback_hours:target_idx + 1]

            # Require a true uninterrupted hourly history ending at the prediction time.
            if not (window_times.diff().dropna() == expected_step).all():
                continue

            if window[feature_columns].isna().any().any():
                continue

            sample = {
                "site_id": site_id,
                "prediction_time": site_df.loc[target_idx, "datetime"],
                "target_streamflow": target_streamflow,
            }

            # For a 168-hour lookback, columns are feature_t-168 ... feature_t-1.
            for offset, (_, history_row) in zip(range(lookback_hours, 0, -1), window.iterrows()):
                for feature in feature_columns:
                    sample[f"{feature}_t-{offset}"] = history_row[feature]

            rows.append(sample)
            if max_samples is not None and len(rows) >= max_samples:
                return pd.DataFrame(rows)

    return pd.DataFrame(rows)


def build_training_parquet_for_site(
    engine,
    target_site_id,
    variable_source_site_ids,
    output_parquet,
    target_variable_match_terms=None,
    lookback_hours=168,
    max_missing_main_predictors=None,
    max_training_samples=None,
    hourly_binning="hour_end",
):
    """Run the full database-to-Parquet training dataset pipeline for one stream site."""
    global MAIN_PREDICTOR_COLUMNS, LOOKBACK_FEATURE_COLUMNS

    check_parquet_support()
    target_variable_match_terms = target_variable_match_terms or TARGET_VARIABLE_MATCH_TERMS

    variables = list_database_variables(engine)
    variable_id_to_name, feature_to_variable_ids, feature_columns = resolve_variable_mapping(
        variables, target_variable_match_terms
    )
    MAIN_PREDICTOR_COLUMNS = feature_columns
    LOOKBACK_FEATURE_COLUMNS = feature_columns + ["year", "month", "day", "hour", "season"]

    availability = list_variable_site_availability(engine)
    print("Available site/variable datastream combinations:")
    display(availability)

    long_df = load_datastream_for_site(
        engine,
        target_site_id,
        variable_id_to_name,
        feature_to_variable_ids,
        feature_columns,
        variable_source_site_ids,
        availability,
    )
    print(f"Loaded long rows: {len(long_df):,}")
    display(long_df.head())

    hourly = long_to_hourly_wide(long_df, feature_columns, binning=hourly_binning)
    hourly = add_time_features(hourly)
    print(f"Hourly rows before sparse-row drop: {len(hourly):,}")
    display(hourly.head())

    prepared = drop_sparse_rows_and_impute(hourly, feature_columns, max_missing_main_predictors)
    print(f"Hourly rows after sparse-row drop and feature imputation: {len(prepared):,}")
    display(prepared.head())

    training = build_supervised_samples(
        prepared,
        lookback_hours,
        LOOKBACK_FEATURE_COLUMNS,
        max_samples=max_training_samples,
    )
    if training.empty:
        raise ValueError(
            "No training samples were created. Check TARGET_SITE_ID, VARIABLE_SOURCE_SITE_IDS, "
            "or increase MAX_MISSING_MAIN_PREDICTORS."
        )

    output_parquet = Path(output_parquet)
    write_parquet_with_pyarrow(training, output_parquet)
    print(f"Saved {len(training):,} inspection samples and {training.shape[1]:,} columns to {output_parquet}")
    return training, prepared


## 4. Build and Save the Training Parquet

In [21]:
training_df, hourly_prepared_df = build_training_parquet_for_site(
    engine=engine,
    target_site_id=TARGET_SITE_ID,
    variable_source_site_ids=VARIABLE_SOURCE_SITE_IDS,
    output_parquet=OUTPUT_PARQUET,
    target_variable_match_terms=TARGET_VARIABLE_MATCH_TERMS,
    lookback_hours=LOOKBACK_HOURS,
    max_missing_main_predictors=MAX_MISSING_MAIN_PREDICTORS,
    max_training_samples=SAMPLE_TRAINING_ROWS,
    hourly_binning=HOURLY_BINNING,
)

display(training_df.head())


Variables available in public.variable and included as features:


,variable_id,variable_code,variable_name,feature_name,unit_name,unit_symbol
0,1,SnowDepth,Snow depth,snow_depth,centimeter,cm
1,2,WaterTemp,Water Temperature,water_temp,degree Celsius,C
2,3,ODO,"Oxygen, dissolved",oxygen_dissolved,milligrams per liter,mg/L
3,4,Discharge,Discharge,streamflow,cubic feet per second,cfs
4,5,AirTemp,Air Temperature,air_temp,degree Celsius,C
5,6,Precip,Precipitation,precipitation,centimeter,cm


Feature columns: ['snow_depth', 'water_temp', 'oxygen_dissolved', 'streamflow', 'air_temp', 'precipitation']
Available site/variable datastream combinations:


,site_id,site_code,variable_id,variable_code,variable_name,row_count,min_datetime,max_datetime
0,1,LR_TG_C,1,SnowDepth,Snow depth,423331,2014-02-19 21:15:00,2026-03-24 17:00:00
1,3,LR_WaterLab_AA,2,WaterTemp,Water Temperature,435460,2013-10-02 20:15:00,2026-03-24 19:00:00
2,4,LR_Mendon_AA,2,WaterTemp,Water Temperature,436398,2013-10-11 22:15:00,2026-03-24 19:00:00
3,3,LR_WaterLab_AA,3,ODO,"Oxygen, dissolved",435148,2013-10-02 20:15:00,2026-03-24 18:00:00
4,3,LR_WaterLab_AA,4,Discharge,Discharge,428202,2014-01-01 07:00:00,2026-03-24 06:00:00
5,4,LR_Mendon_AA,4,Discharge,Discharge,428591,2014-01-01 07:15:00,2026-03-24 19:00:00
6,1,LR_TG_C,5,AirTemp,Air Temperature,320421,2017-02-01 19:45:00,2026-03-24 17:00:00
7,2,LR_GC_C,5,AirTemp,Air Temperature,321544,2017-01-20 23:00:00,2026-03-24 18:00:00
8,2,LR_GC_C,6,Precip,Precipitation,838768,2014-03-18 15:30:00,2026-03-24 17:00:00


Converted 13,990 negative snow_depth readings to missing values
Loaded 423,331 rows for snow_depth from source_site_id=1
Converted 5,599 negative water_temp readings to missing values
Loaded 435,460 rows for water_temp from source_site_id=3
Converted 5,292 negative oxygen_dissolved readings to missing values
Loaded 435,148 rows for oxygen_dissolved from source_site_id=3
Converted 11,900 negative streamflow readings to missing values
Loaded 428,202 rows for streamflow from source_site_id=3
Converted 76,787 negative air_temp readings to missing values
Loaded 321,544 rows for air_temp from source_site_id=2
Converted 208 negative precipitation readings to missing values
Loaded 838,768 rows for precipitation from source_site_id=2
Loaded long rows: 2,882,453


...

,site_id,datetime,snow_depth,water_temp,oxygen_dissolved,streamflow,air_temp,precipitation,year,month,day,hour,season
0,3,2013-10-02 21:00:00,NaN,9.48,9.95,NaN,NaN,NaN,2013,10,2,21,3
1,3,2013-10-02 22:00:00,NaN,9.46,10.03,NaN,NaN,NaN,2013,10,2,22,3
2,3,2013-10-02 23:00:00,NaN,9.46,9.96,NaN,NaN,NaN,2013,10,2,23,3
3,3,2013-10-03 00:00:00,NaN,9.36,10.08,NaN,NaN,NaN,2013,10,3,0,3
4,3,2013-10-03 01:00:00,NaN,9.32,10.08,NaN,NaN,NaN,2013,10,3,1,3


Hourly rows after sparse-row drop and feature imputation: 109,343


,datetime,site_id,snow_depth,water_temp,oxygen_dissolved,streamflow,air_temp,precipitation,year,month,day,hour,season,streamflow_observed
0,2013-10-02 21:00:00,3,72.71,9.48,9.95,67.223206,0.843,0.003658,2013,10,2,21,3,NaN
1,2013-10-02 22:00:00,3,72.71,9.46,10.03,67.223206,0.843,0.003658,2013,10,2,22,3,NaN
2,2013-10-02 23:00:00,3,72.71,9.46,9.96,67.223206,0.843,0.003658,2013,10,2,23,3,NaN
3,2013-10-03 00:00:00,3,72.71,9.36,10.08,67.223206,0.843,0.003658,2013,10,3,0,3,NaN
4,2013-10-03 01:00:00,3,72.71,9.32,10.08,67.223206,0.843,0.003658,2013,10,3,1,3,NaN


Saved 104,432 inspection samples and 1,851 columns to streamflow_training_site_3_full.parquet


,site_id,prediction_time,target_streamflow,snow_depth_t-168,water_temp_t-168,oxygen_dissolved_t-168,streamflow_t-168,air_temp_t-168,precipitation_t-168,year_t-168,month_t-168,day_t-168,hour_t-168,season_t-168,snow_depth_t-167,water_temp_t-167,oxygen_dissolved_t-167,streamflow_t-167,air_temp_t-167,precipitation_t-167,year_t-167,month_t-167,day_t-167,hour_t-167,season_t-167,snow_depth_t-166,water_temp_t-166,oxygen_dissolved_t-166,streamflow_t-166,air_temp_t-166,precipitation_t-166,year_t-166,month_t-166,day_t-166,hour_t-166,season_t-166,snow_depth_t-165,water_temp_t-165,oxygen_dissolved_t-165,streamflow_t-165,air_temp_t-165,precipitation_t-165,year_t-165,month_t-165,day_t-165,hour_t-165,season_t-165,snow_depth_t-164,water_temp_t-164,oxygen_dissolved_t-164,streamflow_t-164,air_temp_t-164,precipitation_t-164,year_t-164,month_t-164,day_t-164,hour_t-164,season_t-164,snow_depth_t-163,water_temp_t-163,...,year_t-6,month_t-6,day_t-6,hour_t-6,season_t-6,snow_depth_t-5,water_temp_t-5,oxygen_dissolved_t-5,streamflow_t-5,air_temp_t-5,precipitation_t-5,year_t-5,month_t-5,day_t-5,hour_t-5,season_t-5,snow_depth_t-4,water_temp_t-4,oxygen_dissolved_t-4,streamflow_t-4,air_temp_t-4,precipitation_t-4,year_t-4,month_t-4,day_t-4,hour_t-4,season_t-4,snow_depth_t-3,water_temp_t-3,oxygen_dissolved_t-3,streamflow_t-3,air_temp_t-3,precipitation_t-3,year_t-3,month_t-3,day_t-3,hour_t-3,season_t-3,snow_depth_t-2,water_temp_t-2,oxygen_dissolved_t-2,streamflow_t-2,air_temp_t-2,precipitation_t-2,year_t-2,month_t-2,day_t-2,hour_t-2,season_t-2,snow_depth_t-1,water_temp_t-1,oxygen_dissolved_t-1,streamflow_t-1,air_temp_t-1,precipitation_t-1,year_t-1,month_t-1,day_t-1,hour_t-1,season_t-1
0,3,2014-01-01 07:00:00,67.223206,72.71,3.94,11.56,67.223206,0.843,0.003658,2013,12,25,7,0,72.71,3.79,11.60,67.223206,0.843,0.003658,2013,12,25,8,0,72.71,3.62,11.64,67.223206,0.843,0.003658,2013,12,25,9,0,72.71,3.42,11.70,67.223206,0.843,0.003658,2013,12,25,10,0,72.71,3.21,11.76,67.223206,0.843,0.003658,2013,12,25,11,0,72.71,3.04,...,2014,1,1,1,0,72.71,3.49,11.67,67.223206,0.843,0.003658,2014,1,1,2,0,72.71,3.53,11.65,67.223206,0.843,0.003658,2014,1,1,3,0,72.71,3.58,11.62,67.223206,0.843,0.003658,2014,1,1,4,0,72.71,3.60,11.61,67.223206,0.843,0.003658,2014,1,1,5,0,72.71,3.62,11.59,67.223206,0.843,0.003658,2014,1,1,6,0
1,3,2014-01-01 08:00:00,68.160918,72.71,3.79,11.60,67.223206,0.843,0.003658,2013,12,25,8,0,72.71,3.62,11.64,67.223206,0.843,0.003658,2013,12,25,9,0,72.71,3.42,11.70,67.223206,0.843,0.003658,2013,12,25,10,0,72.71,3.21,11.76,67.223206,0.843,0.003658,2013,12,25,11,0,72.71,3.04,11.81,67.223206,0.843,0.003658,2013,12,25,12,0,72.71,2.87,...,2014,1,1,2,0,72.71,3.53,11.65,67.223206,0.843,0.003658,2014,1,1,3,0,72.71,3.58,11.62,67.223206,0.843,0.003658,2014,1,1,4,0,72.71,3.60,11.61,67.223206,0.843,0.003658,2014,1,1,5,0,72.71,3.62,11.59,67.223206,0.843,0.003658,2014,1,1,6,0,72.71,3.66,11.57,67.223206,0.843,0.003658,2014,1,1,7,0
2,3,2014-01-01 09:00:00,67.037188,72.71,3.62,11.64,67.223206,0.843,0.003658,2013,12,25,9,0,72.71,3.42,11.70,67.223206,0.843,0.003658,2013,12,25,10,0,72.71,3.21,11.76,67.223206,0.843,0.003658,2013,12,25,11,0,72.71,3.04,11.81,67.223206,0.843,0.003658,2013,12,25,12,0,72.71,2.87,11.87,67.223206,0.843,0.003658,2013,12,25,13,0,72.71,2.66,...,2014,1,1,3,0,72.71,3.58,11.62,67.223206,0.843,0.003658,2014,1,1,4,0,72.71,3.60,11.61,67.223206,0.843,0.003658,2014,1,1,5,0,72.71,3.62,11.59,67.223206,0.843,0.003658,2014,1,1,6,0,72.71,3.66,11.57,67.223206,0.843,0.003658,2014,1,1,7,0,72.71,3.71,11.55,68.160918,0.843,0.003658,2014,1,1,8,0
3,3,2014-01-01 10:00:00,66.620491,72.71,3.42,11.70,67.223206,0.843,0.003658,2013,12,25,10,0,72.71,3.21,11.76,67.223206,0.843,0.003658,2013,12,25,11,0,72.71,3.04,11.81,67.223206,0.843,0.003658,2013,12,25,12,0,72.71,2.87,11.87,67.223206,0.843,0.003658,2013,12,25,13,0,72.71,2.66,11.94,67.223206,0.843,0.003658,2013,12,25,14,0,72.71,2.49,...,2014,1,1,4,0,72.71,3.60,11.61,67.223206,0.843,0.003658,2014,1,1,5,0,72.71,3.62,11.59,67.223206,0.843,0.00365

## 5. Verify Saved Parquet

In [17]:
saved_df = read_parquet_with_pyarrow(OUTPUT_PARQUET)
print(saved_df.shape)
display(saved_df.head())

(200, 1851)


,site_id,prediction_time,target_streamflow,snow_depth_t-168,water_temp_t-168,oxygen_dissolved_t-168,streamflow_t-168,air_temp_t-168,precipitation_t-168,year_t-168,month_t-168,day_t-168,hour_t-168,season_t-168,snow_depth_t-167,water_temp_t-167,oxygen_dissolved_t-167,streamflow_t-167,air_temp_t-167,precipitation_t-167,year_t-167,month_t-167,day_t-167,hour_t-167,season_t-167,snow_depth_t-166,water_temp_t-166,oxygen_dissolved_t-166,streamflow_t-166,air_temp_t-166,precipitation_t-166,year_t-166,month_t-166,day_t-166,hour_t-166,season_t-166,snow_depth_t-165,water_temp_t-165,oxygen_dissolved_t-165,streamflow_t-165,air_temp_t-165,precipitation_t-165,year_t-165,month_t-165,day_t-165,hour_t-165,season_t-165,snow_depth_t-164,water_temp_t-164,oxygen_dissolved_t-164,streamflow_t-164,air_temp_t-164,precipitation_t-164,year_t-164,month_t-164,day_t-164,hour_t-164,season_t-164,snow_depth_t-163,water_temp_t-163,...,year_t-6,month_t-6,day_t-6,hour_t-6,season_t-6,snow_depth_t-5,water_temp_t-5,oxygen_dissolved_t-5,streamflow_t-5,air_temp_t-5,precipitation_t-5,year_t-5,month_t-5,day_t-5,hour_t-5,season_t-5,snow_depth_t-4,water_temp_t-4,oxygen_dissolved_t-4,streamflow_t-4,air_temp_t-4,precipitation_t-4,year_t-4,month_t-4,day_t-4,hour_t-4,season_t-4,snow_depth_t-3,water_temp_t-3,oxygen_dissolved_t-3,streamflow_t-3,air_temp_t-3,precipitation_t-3,year_t-3,month_t-3,day_t-3,hour_t-3,season_t-3,snow_depth_t-2,water_temp_t-2,oxygen_dissolved_t-2,streamflow_t-2,air_temp_t-2,precipitation_t-2,year_t-2,month_t-2,day_t-2,hour_t-2,season_t-2,snow_depth_t-1,water_temp_t-1,oxygen_dissolved_t-1,streamflow_t-1,air_temp_t-1,precipitation_t-1,year_t-1,month_t-1,day_t-1,hour_t-1,season_t-1
0,3,2014-01-01 07:00:00,67.223206,72.71,3.94,11.56,67.223206,0.843,0.003658,2013,12,25,7,0,72.71,3.79,11.60,67.223206,0.843,0.003658,2013,12,25,8,0,72.71,3.62,11.64,67.223206,0.843,0.003658,2013,12,25,9,0,72.71,3.42,11.70,67.223206,0.843,0.003658,2013,12,25,10,0,72.71,3.21,11.76,67.223206,0.843,0.003658,2013,12,25,11,0,72.71,3.04,...,2014,1,1,1,0,72.71,3.49,11.67,67.223206,0.843,0.003658,2014,1,1,2,0,72.71,3.53,11.65,67.223206,0.843,0.003658,2014,1,1,3,0,72.71,3.58,11.62,67.223206,0.843,0.003658,2014,1,1,4,0,72.71,3.60,11.61,67.223206,0.843,0.003658,2014,1,1,5,0,72.71,3.62,11.59,67.223206,0.843,0.003658,2014,1,1,6,0
1,3,2014-01-01 08:00:00,68.160918,72.71,3.79,11.60,67.223206,0.843,0.003658,2013,12,25,8,0,72.71,3.62,11.64,67.223206,0.843,0.003658,2013,12,25,9,0,72.71,3.42,11.70,67.223206,0.843,0.003658,2013,12,25,10,0,72.71,3.21,11.76,67.223206,0.843,0.003658,2013,12,25,11,0,72.71,3.04,11.81,67.223206,0.843,0.003658,2013,12,25,12,0,72.71,2.87,...,2014,1,1,2,0,72.71,3.53,11.65,67.223206,0.843,0.003658,2014,1,1,3,0,72.71,3.58,11.62,67.223206,0.843,0.003658,2014,1,1,4,0,72.71,3.60,11.61,67.223206,0.843,0.003658,2014,1,1,5,0,72.71,3.62,11.59,67.223206,0.843,0.003658,2014,1,1,6,0,72.71,3.66,11.57,67.223206,0.843,0.003658,2014,1,1,7,0
2,3,2014-01-01 09:00:00,67.037188,72.71,3.62,11.64,67.223206,0.843,0.003658,2013,12,25,9,0,72.71,3.42,11.70,67.223206,0.843,0.003658,2013,12,25,10,0,72.71,3.21,11.76,67.223206,0.843,0.003658,2013,12,25,11,0,72.71,3.04,11.81,67.223206,0.843,0.003658,2013,12,25,12,0,72.71,2.87,11.87,67.223206,0.843,0.003658,2013,12,25,13,0,72.71,2.66,...,2014,1,1,3,0,72.71,3.58,11.62,67.223206,0.843,0.003658,2014,1,1,4,0,72.71,3.60,11.61,67.223206,0.843,0.003658,2014,1,1,5,0,72.71,3.62,11.59,67.223206,0.843,0.003658,2014,1,1,6,0,72.71,3.66,11.57,67.223206,0.843,0.003658,2014,1,1,7,0,72.71,3.71,11.55,68.160918,0.843,0.003658,2014,1,1,8,0
3,3,2014-01-01 10:00:00,66.620491,72.71,3.42,11.70,67.223206,0.843,0.003658,2013,12,25,10,0,72.71,3.21,11.76,67.223206,0.843,0.003658,2013,12,25,11,0,72.71,3.04,11.81,67.223206,0.843,0.003658,2013,12,25,12,0,72.71,2.87,11.87,67.223206,0.843,0.003658,2013,12,25,13,0,72.71,2.66,11.94,67.223206,0.843,0.003658,2013,12,25,14,0,72.71,2.49,...,2014,1,1,4,0,72.71,3.60,11.61,67.223206,0.843,0.003658,2014,1,1,5,0,72.71,3.62,11.59,67.223206,0.843,0.00365